# Generating cutflow spreadsheets
Analysis algorithms often print out the cutflow table to the terminal.

The accompanying file `cutflow_funcs.py` creates some simple functions to achieve some basic goals:
1. Extract a cutflow string from a text file.
2. Convert an extracted string to a DF.
3. Manipulate the DF (add columns, change column names, etc.)
4. Save DFs to a spreadsheet.

This notebook is meant as a playground to use these functions on real cutflows.

Section 1 gives in an example use of the functions.

NOTE: analyses often use different cutflow formats - you may have to define your own extraction and conversion functions.

Imports

In [51]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import custom functions
import re
import importlib
imported_module = importlib.import_module("cutflow_funcs")
importlib.reload(imported_module)

# Help check you are using the latest version
import cutflow_funcs as cf
cf.test_func()

1.9

***
## 1. Example use - DO NOT CHANGE
***
Demonstration of the functions on cutflows in the ZdZdPostProcessing format.

In [9]:
# Examples
cutflow_dir = '/Users/matt/Documents/work/0-PhD/particle/xxzx-analysis/code_ZdZd/cutflows/'

### 1.1 ZdZdPostProcessing cutflows

In [52]:
# File of ZdZdPP cutflows
zdzdPP_cutflows_2026_02 = '2026-02-09_ZdZdPP-cutflows.txt'

# Extract from the file
zdzdPP_cutflow_dict = cf.parse_ZdZdPP_cutflow_file(cutflow_dir + zdzdPP_cutflows_2026_02)

# Save DFs using a dictionary comprehension
zdzdPP_cutflow_dfs = {name: cf.simplify_ZdZdPP_cutflow(cf.str_to_df_ZdZdPP(cutflow))
                      for name, cutflow in zdzdPP_cutflow_dict.items()}
# Save to spreadsheet
# cf.make_spreadsheet(cutflow_dir+'2026-02-14_ZdZdPP-cutflows.xlsx', zdzdPP_cutflow_dfs)

# cf.simplify_ZdZdPP_cutflow(cf.str_to_df_ZdZdPP(zdzdPP_cutflow_dict['mc23a_zd30']))
# display(zdzdPP_cutflow_dfs['mc23a_zd30'])

# Print the data types of the columns in the cutflow dataframe
print(zdzdPP_cutflow_dfs['mc23a_zd30'].dtypes)

Cuts             object
weights_4e      float64
events_4e         int64
weights_2e2m    float64
events_2e2m       int64
weights_4m      float64
events_4m         int64
weights_All     float64
events_All        int64
dtype: object


### 1.2 Scott's cutflows

In [12]:
# File of Scott's cutflows
scott_cutflows_2025_10 = '2025-10-17_Scott-ZdZd-R3-cutflows.txt'
# Extract from the file
scotts_cutflow_dict = cf.parse_Scott_cutflow_file(cutflow_dir + scott_cutflows_2025_10)
# Save DFs using a dictionary comprehension
scotts_cutflow_dfs = {name.replace(' --- ', '_'): cf.str_to_df_Scott(cutflow)
                      for name, cutflow in scotts_cutflow_dict.items()}
# Save to spreadsheet
# make_spreadsheet('test4.xlsx', scotts_cutflow_dfs)

scotts_cutflow_dfs['zd5_23a_reco']

,Cut,N4e,Wt4e,Nem,Wtem,N4m,Wt4m,N all,Wt all
0,weight,,,,,,,22846,0.0000
1,overlap,,,,,,,22846,0.0000
2,clean,,,,,,,22846,0.0000
3,pv,,,,,,,22846,0.0000
4,trigger,,,,,,,19443,0.0000
5,jetclean,,,,,,,19443,0.0000
6,*Quad selection*,,,,,,,,
7,quad sfos,,,,,,,19104,0.0000
8,quad or,,,,,,,19103,0.0000
9,quad kin,,,,,,,18378,0.0000


***
## 2. Your code here!
***

In [13]:
# Path to the cutflow files

# Extract the cutflow data from the files

# Convert the cutflow data into DataFrames

# Manipulate the DataFrames as needed (e.g., simplify, calculate efficiencies)

# Save the DataFrames to a spreadsheet

***
## 3. Scott's cutflows vs. ZdZdPostProcessing
***

Fixing ZdZdPP

In [58]:
new_col_order = ['Cuts', 'events_4e', 'weights_4e', 'events_2e2m', 'weights_2e2m', 'events_4m', 'weights_4m', 'events_All', 'weights_All']

zdzdPP_cutflow_dfs_copy = zdzdPP_cutflow_dfs.copy()

# Drop rows where the value of 'Cut' is 'MuType'
for name, df in zdzdPP_cutflow_dfs_copy.items():
    zdzdPP_cutflow_dfs_copy[name] = df[df['Cuts'] != 'MuType'].reset_index(drop=True, inplace=False).replace(0, np.nan)[new_col_order]

# zdzdPP_cutflow_dfs_copy['mc23e_zd30'].reset_index(drop=True, inplace=True)
zdzdPP_cutflow_dfs_copy['mc23a_zd30'].columns
# Reorder the columns in the DataFrame
# for name, df in zdzdPP_cutflow_dfs_copy.items():
#     zdzdPP_cutflow_dfs_copy[name] = df[new_col_order]

zdzdPP_cutflow_dfs_copy['mc23d_zd30']

# Save to spreadsheet
cf.make_spreadsheet(cutflow_dir+'2026-02-17_ZdZdPP-cutflows.xlsx', zdzdPP_cutflow_dfs_copy)

Fixing Scott's cutflows

In [68]:
def simplify_Scott_cutflow(df, new_cols):
    # Find the index of the row where the value of 'Cut' is '*AS SR1*'
    index = df[df['Cut'] == '*AS SR1*'].index
    # Keep only the rows up to and including that index
    df = df.loc[:index[0]]
    # Remove rows where 'Cut' contains 'overlap', 'jpsi' or '*'
    simplified_df = df[~df['Cut'].str.contains('overlap|jetclean|jpsi|tight|\*', case=False, regex=True)]\
        .reset_index(drop=True, inplace=False)#[new_col_order]
    # Rename the columns
    simplified_df.columns = new_cols

    return simplified_df

In [71]:
simplify_Scott_cutflow(scotts_cutflow_dfs['zd30_23a_reco'], new_col_order)

# scotts_cutflow_dfs['zd30_23d_reco'].columns = new_col_order
scotts_cutflow_dfs['zd30_23d_reco']

,Cuts,events_4e,weights_4e,events_2e2m,weights_2e2m,events_4m,weights_4m,events_All,weights_All
0,weight,,,,,,,50000,0.0000
1,overlap,,,,,,,50000,0.0000
2,clean,,,,,,,50000,0.0000
3,pv,,,,,,,50000,0.0000
4,trigger,,,,,,,38800,0.0000
5,jetclean,,,,,,,38459,0.0000
6,*Quad selection*,,,,,,,,
7,quad sfos,,,,,,,17306,0.0000
8,quad or,,,,,,,17302,0.0000
9,quad kin,,,,,,,16830,0.0000


In [29]:
scott_copy = scotts_cutflow_dfs.copy()
# Remove all rows from scott_copy['zd30_23d_reco'] where the value of 'Cut' contains 'overlap', 'jpsi' or '*'
# test_scott_df = scott_copy['zd30_23d_reco'][~scott_copy['zd30_23d_reco']['Cut'].str.contains('overlap|jpsi|\*', case=False, regex=True)]
# Find the index of the row where the value of 'Cut' is '*AS SR1*'
test_scott_df = scott_copy['zd30_23d_reco'][scott_copy['zd30_23d_reco']['Cut'] == '*AS SR1*']
test_scott_df.index
# Get the subset of the df up until this index
test_scott_df_subset = scott_copy['zd30_23d_reco'].iloc[:test_scott_df.index[0]]
test_scott_df_subset

# new_scott_df = {}
# for name, df in scotts_cutflow_dfs.items():
#     new_scott_df[name] = df[~df['Cut'].str.contains('overlap|jpsi|\*', case=False, regex=True)]

,Cut,N4e,Wt4e,Nem,Wtem,N4m,Wt4m,N all,Wt all
0,weight,,,,,,,50000,0.0000
1,overlap,,,,,,,50000,0.0000
2,clean,,,,,,,50000,0.0000
3,pv,,,,,,,50000,0.0000
4,trigger,,,,,,,38800,0.0000
5,jetclean,,,,,,,38459,0.0000
6,*Quad selection*,,,,,,,,
7,quad sfos,,,,,,,17306,0.0000
8,quad or,,,,,,,17302,0.0000
9,quad kin,,,,,,,16830,0.0000


## 4. Scott's cutflows - updated

2026-02-17: Removed jet cleaning, added weights and added scale factors for R3 electrons.

In [3]:
new_mc23a_zd30 = '''|                 |N4e  |Wt4e   |Nem  |Wtem   |N4m  |Wt4m   |N all |Wt all |
|*Precuts*                                                         |||||||||
|weight           |     |       |     |       |     |       |60000 |60000  |
|overlap          |     |       |     |       |     |       |60000 |60000  |
|clean            |     |       |     |       |     |       |60000 |60000  |
|pv               |     |       |     |       |     |       |60000 |60000  |
|trigger          |     |       |     |       |     |       |51707 |51714  |
|*Quad selection*                                                  |||||||||
|quad sfos        |     |       |     |       |     |       |23636 |23684  |
|quad or          |     |       |     |       |     |       |23606 |23654  |
|quad kin         |     |       |     |       |     |       |22900 |22942  |
|quad trigmatch   |     |       |     |       |     |       |22768 |22808  |
|quad dr          |     |       |     |       |     |       |22736 |22776  |
|*Common*                                                          |||||||||
|isol             |3010 |2846.8 |8948 |8351.4 |6954 |6366.6 |18912 |17565  |
|e_id             |3010 |2846.8 |8948 |8351.4 |6954 |6366.6 |18912 |17565  |
|impact           |2990 |2828.7 |8763 |8180.8 |6750 |6179.1 |18503 |17189  |
|jpsi             |2990 |2828.7 |8762 |8179.8 |6749 |6178.2 |18501 |17187  |
|upsilon          |2986 |2825.6 |8761 |8178.7 |6725 |6156.1 |18472 |17160  |
|lowmveto         |2986 |2825.6 |8759 |8177.1 |6724 |6155.1 |18469 |17158  |
|*HM*                                                              |||||||||
|hwind            |2601 |2459.1 |8032 |7492.0 |6456 |5908.2 |17089 |15859  |
|zveto            |1817 |1715.3 |8032 |7492.0 |4359 |3997.3 |14208 |13205  |
|loose            |1810 |1708.8 |8023 |7483.7 |4326 |3967.1 |14159 |13160  |
|medium           |1802 |1701.3 |7992 |7456.3 |4321 |3963.1 |14115 |13121  |
|tight            |521  |495.68 |2390 |2244.0 |1557 |1425.7 |4468  |4165.4 |
|*AS SR1*                                                          |||||||||
|SR1              |114  |109.50 |331  |312.00 |228  |209.87 |673   |631.37 |
|zveto            |108  |104.42 |331  |312.00 |210  |193.52 |649   |609.95 |
|loose            |90   |87.371 |275  |259.23 |169  |155.20 |534   |501.80 |
|medium           |69   |67.340 |204  |191.25 |123  |116.28 |396   |374.87 |
|tight            |39   |37.843 |71   |67.378 |46   |44.507 |156   |149.73 |
|*AS SR2*                                                          |||||||||
|SR2              |271  |256.96 |396  |373.15 |40   |37.030 |707   |667.14 |
|zveto            |244  |229.96 |396  |373.15 |38   |34.896 |678   |638.00 |
|loose            |242  |228.09 |396  |373.15 |37   |34.382 |675   |635.62 |
|medium           |224  |211.05 |364  |344.73 |37   |34.382 |625   |590.16 |
|tight            |182  |172.51 |279  |265.27 |37   |34.382 |498   |472.17 |'''

In [6]:
new_mc23a_zd30_df = cf.str_to_df_Scott(new_mc23a_zd30)
new_mc23a_zd30_df

,Cut,N4e,Wt4e,Nem,Wtem,N4m,Wt4m,N all,Wt all
0,weight,,,,,,,60000,60000
1,overlap,,,,,,,60000,60000
2,clean,,,,,,,60000,60000
3,pv,,,,,,,60000,60000
4,trigger,,,,,,,51707,51714
5,*Quad selection*,,,,,,,,
6,quad sfos,,,,,,,23636,23684
7,quad or,,,,,,,23606,23654
8,quad kin,,,,,,,22900,22942
9,quad trigmatch,,,,,,,22768,22808


In [10]:
cf.make_spreadsheet(cutflow_dir + 'test5.xlsx', {'new_mc23a_zd30': new_mc23a_zd30_df})